In [0]:
%sql
SELECT * FROM `workspace`.`default`.`diabetes_raw_practice_dataset`;

patient_id,age,sex,bmi,waist_cm,systolic_bp,diastolic_bp,fasting_glucose_mgdl,hba1c_percent,total_chol_mgdl,hdl_mgdl,ldl_mgdl,triglycerides_mgdl,smoker,physical_activity_mins_week,family_history_diabetes,diabetes_dx
P0001,45,female,31.2,98,138,86,132,7.1,210,42,130,185,No,60,Yes,Yes
P0002,29,M,24.5,82,118,76,92,5.4,170,55,95,110,no,180,N,No
P0003,56,MALE,33.8,110,152,92,168,8.4,240,38,145,260,YES,30,Yes,Y
P0004,40,F,27.1,90,126,80,105,6.0,195,50,120,140,No,120,Yes,No
P0005,63,F,29.9,102,146,88,154,7.8,225,44,135,210,,45,Yes,Yes
P0006,35,m,26.3,88,122,78,99,5.6,180,48,110,130,Y,90,No,No
P0007,51,F,34.6,112,160,96,190,9.2,255,36,155,300,No,20,YES,Yes
P0008,47,M,28.7,96,134,84,128,6.8,205,46,125,175,No,75,Yes,Yes
P0009,38,F,23.9,80,114,72,89,5.3,165,58,90,95,No,210,No,No
P0010,59,M,30.5,104,148,90,160,8.1,235,40,140,240,Yes,40,Yes,Yes


In [0]:
-- Total rows
SELECT COUNT(*) AS total_rows
FROM workspace.default.diabetes_raw_practice_dataset;

-- Duplicate patient_id
SELECT TRIM(patient_id) AS patient_id, COUNT(*) AS cnt
FROM workspace.default.diabetes_raw_practice_dataset
GROUP BY TRIM(patient_id)
HAVING COUNT(*) > 1;

-- Missing/blank patient_id
SELECT COUNT(*) AS blank_patient_id
FROM workspace.default.diabetes_raw_practice_dataset
WHERE patient_id IS NULL OR TRIM(patient_id) = '';

blank_patient_id
1


In [0]:
CREATE OR REPLACE TABLE workspace.default.diabetes_clean AS
WITH base AS (
  SELECT
    TRIM(patient_id) AS patient_id,

    CASE WHEN CAST(age AS INT) BETWEEN 0 AND 120 THEN CAST(age AS INT) ELSE NULL END AS age,

    CASE
      WHEN LOWER(TRIM(sex)) IN ('f','female') THEN 'F'
      WHEN LOWER(TRIM(sex)) IN ('m','male') THEN 'M'
      ELSE 'Unknown'
    END AS sex,

    CASE
      WHEN LOWER(TRIM(bmi)) IN ('n/a','na','') THEN NULL
      ELSE CAST(TRIM(bmi) AS DOUBLE)
    END AS bmi,

    CASE WHEN TRIM(waist_cm) = '' THEN NULL ELSE CAST(TRIM(waist_cm) AS DOUBLE) END AS waist_cm,
    CAST(systolic_bp AS INT) AS systolic_bp,
    CAST(diastolic_bp AS INT) AS diastolic_bp,

    CASE WHEN TRIM(fasting_glucose_mgdl) = '' THEN NULL ELSE CAST(fasting_glucose_mgdl AS DOUBLE) END AS fasting_glucose_mgdl,

    CASE
      WHEN hba1c_percent LIKE '%,%' THEN CAST(REPLACE(hba1c_percent, ',', '.') AS DOUBLE)
      WHEN TRIM(hba1c_percent) = '' THEN NULL
      ELSE CAST(hba1c_percent AS DOUBLE)
    END AS hba1c_percent,

    CAST(total_chol_mgdl AS DOUBLE) AS total_chol_mgdl,
    CAST(hdl_mgdl AS DOUBLE) AS hdl_mgdl,
    CAST(ldl_mgdl AS DOUBLE) AS ldl_mgdl,
    CAST(triglycerides_mgdl AS DOUBLE) AS triglycerides_mgdl,

    CASE
      WHEN LOWER(TRIM(smoker)) IN ('yes','y','true','1') THEN 'Yes'
      WHEN LOWER(TRIM(smoker)) IN ('no','n','false','0') THEN 'No'
      ELSE 'Unknown'
    END AS smoker,

    CASE
      WHEN physical_activity_mins_week RLIKE '^[0-9]+$' THEN CAST(physical_activity_mins_week AS INT)
      ELSE NULL
    END AS physical_activity_mins_week,

    CASE
      WHEN LOWER(TRIM(family_history_diabetes)) IN ('yes','y','true','1') THEN 'Yes'
      WHEN LOWER(TRIM(family_history_diabetes)) IN ('no','n','false','0') THEN 'No'
      ELSE 'Unknown'
    END AS family_history_diabetes,

    CASE
      WHEN LOWER(TRIM(diabetes_dx)) IN ('yes','y','true','1') THEN 'Yes'
      WHEN LOWER(TRIM(diabetes_dx)) IN ('no','n','false','0') THEN 'No'
      ELSE 'Unknown'
    END AS diabetes_dx
  FROM workspace.default.diabetes_raw_practice_dataset
  WHERE patient_id IS NOT NULL AND TRIM(patient_id) <> ''
),
dedup AS (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY patient_id) AS rn
  FROM base
)
SELECT * EXCEPT(rn)
FROM dedup
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
SELECT * FROM workspace.default.diabetes_clean LIMIT 50;

SELECT COUNT(*) AS clean_rows
FROM workspace.default.diabetes_clean;

clean_rows
26


In [0]:
CREATE OR REPLACE TABLE workspace.default.diabetes_gold AS
SELECT *,
  CASE
    WHEN bmi >= 30 THEN 'Obese'
    WHEN bmi >= 25 THEN 'Overweight'
    WHEN bmi >= 18.5 THEN 'Normal'
    WHEN bmi IS NULL THEN 'Unknown'
    ELSE 'Underweight'
  END AS bmi_category,

  CASE
    WHEN hba1c_percent >= 6.5 THEN 'Diabetes range'
    WHEN hba1c_percent >= 5.7 THEN 'Prediabetes range'
    WHEN hba1c_percent IS NULL THEN 'Unknown'
    ELSE 'Normal'
  END AS a1c_band
FROM workspace.default.diabetes_clean;workspace.default.diabetes_cleanworkspace.default.diabetes_clean

num_affected_rows,num_inserted_rows


In [0]:

SELECT * 
FROM workspace.default.diabetes_gold;

patient_id,age,sex,bmi,waist_cm,systolic_bp,diastolic_bp,fasting_glucose_mgdl,hba1c_percent,total_chol_mgdl,hdl_mgdl,ldl_mgdl,triglycerides_mgdl,smoker,physical_activity_mins_week,family_history_diabetes,diabetes_dx,bmi_category,a1c_band
P0001,45,F,31.2,98.0,138,86,132.0,7.1,210.0,42.0,130.0,185.0,No,60,Yes,Yes,Obese,Diabetes range
P0002,29,M,24.5,82.0,118,76,92.0,5.4,170.0,55.0,95.0,110.0,No,180,No,No,Normal,Normal
P0003,56,M,33.8,110.0,152,92,168.0,8.4,240.0,38.0,145.0,260.0,Yes,30,Yes,Yes,Obese,Diabetes range
P0004,40,F,27.1,90.0,126,80,105.0,6.0,195.0,50.0,120.0,140.0,No,120,Yes,No,Overweight,Prediabetes range
P0005,63,F,29.9,102.0,146,88,154.0,7.8,225.0,44.0,135.0,210.0,Unknown,45,Yes,Yes,Overweight,Diabetes range
P0006,35,M,26.3,88.0,122,78,99.0,5.6,180.0,48.0,110.0,130.0,Yes,90,No,No,Overweight,Normal
P0007,51,F,34.6,112.0,160,96,190.0,9.2,255.0,36.0,155.0,300.0,No,20,Yes,Yes,Obese,Diabetes range
P0008,47,M,28.7,96.0,134,84,128.0,6.8,205.0,46.0,125.0,175.0,No,75,Yes,Yes,Overweight,Diabetes range
P0009,38,F,23.9,80.0,114,72,89.0,5.3,165.0,58.0,90.0,95.0,No,210,No,No,Normal,Normal
P0010,59,M,30.5,104.0,148,90,160.0,8.1,235.0,40.0,140.0,240.0,Yes,40,Yes,Yes,Obese,Diabetes range


In [0]:
%sql
SELECT 
    a1c_band,
    COUNT(*) AS patient_count
FROM workspace.default.diabetes_gold
GROUP BY a1c_band
ORDER BY patient_count DESC;

a1c_band,patient_count
Diabetes range,16
Normal,5
Prediabetes range,5


In [0]:
%sql
SELECT 
    diabetes_dx,
    COUNT(*) AS patient_count
FROM workspace.default.diabetes_gold
GROUP BY diabetes_dx;

diabetes_dx,patient_count
Yes,15
No,10
Unknown,1


In [0]:
%sql
SELECT 
    bmi_category,
    COUNT(*) AS total_patients
FROM workspace.default.diabetes_gold
GROUP BY bmi_category
ORDER BY total_patients DESC;


bmi_category,total_patients
Overweight,13
Obese,9
Normal,3
Unknown,1


In [0]:
%sql
SELECT 
    a1c_band,
    COUNT(*) AS patient_count
FROM workspace.default.diabetes_gold
GROUP BY a1c_band
ORDER BY patient_count DESC;

a1c_band,patient_count
Diabetes range,16
Normal,5
Prediabetes range,5


Databricks visualization. Run in Databricks to view.